# 22.2427 | Stretching good packings to nearby n

The file at the end scores 22.2427258 with the official metric. The best public file I could find this morning (6 Nov) is at 22.24987, so this one is 0.0071 lower, and almost all of that comes from one idea: when a packing is already about as good as the best known for its size, cut it, copy one column and one row, and let `fq` settle the result. That, and a longer relaxation of the stretches that nearly won, improved 52 puzzles.

Below: n = 930, the same packing with one column and one row copied in (993 squares, the copies outlined), and the 993 that `fq` made of it, 0.008 smaller than the 993 I had.

In [ ]:
import csv, glob, math, os, shutil, subprocess, sys, time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from scipy.spatial import cKDTree

DATA = Path(glob.glob('/kaggle/input/**/stretch_wins.csv', recursive=True)[0]).parent
BANK = glob.glob('/kaggle/input/**/perpuzzle_best.csv', recursive=True)[0]
WORK = Path('/kaggle/working')
FQ = '/tmp/fq'                     # outside /kaggle/working, so the binary is not part of the notebook's output
shutil.copy(DATA / 'fq', FQ)
os.chmod(FQ, 0o755)
sys.path.insert(0, str(DATA))
sys.path.insert(0, '/tmp')

OFF = np.array([(-.5, -.5), (.5, -.5), (.5, .5), (-.5, .5)])


def corners(s):
    s = np.asarray(s, float)
    c, si = np.cos(s[:, 2]), np.sin(s[:, 2])
    x = s[:, None, 0] + OFF[None, :, 0] * c[:, None] - OFF[None, :, 1] * si[:, None]
    y = s[:, None, 1] + OFF[None, :, 0] * si[:, None] + OFF[None, :, 1] * c[:, None]
    return np.stack([x, y], -1)


def normalise(s):
    s = np.array(s, float)
    s[:, :2] -= corners(s).reshape(-1, 2).min(0)
    return s


def side(s):
    p = corners(s).reshape(-1, 2)
    return float(max(np.ptp(p[:, 0]), np.ptp(p[:, 1])))


def tilt_deg(t):
    return (np.degrees(t) % 90 + 45) % 90 - 45


def overlaps(s, tol=1e-9):
    """Number of pairs overlapping by more than tol (separating-axis test on all close pairs)."""
    s = np.asarray(s, float)
    cs = corners(s)
    p = cKDTree(s[:, :2]).query_pairs(math.sqrt(2) + 1e-6, output_type='ndarray')
    if len(p) == 0:
        return 0
    I, J = p[:, 0], p[:, 1]
    ci, si, cj, sj = np.cos(s[I, 2]), np.sin(s[I, 2]), np.cos(s[J, 2]), np.sin(s[J, 2])
    axes = np.stack([np.stack([ci, si], -1), np.stack([-si, ci], -1), np.stack([cj, sj], -1), np.stack([-sj, cj], -1)], 1)
    pi, pj = np.einsum('mkd,mad->mak', cs[I], axes), np.einsum('mkd,mad->mak', cs[J], axes)
    sep = (np.maximum(pi.min(-1), pj.min(-1)) - np.minimum(pi.max(-1), pj.max(-1))).max(-1)
    return int((sep < -tol).sum())


def draw(ax, s, title='', mark=None):
    """Red = empty space, grey = axis-aligned squares, colour = tilt angle; `mark` (bool mask) gets a black outline."""
    s = normalise(s)
    L = side(s)
    ax.add_patch(plt.Rectangle((0, 0), L, L, color='#d9534f'))
    a = tilt_deg(s[:, 2])
    cols = ['#d4d4d4' if abs(v) < 0.05 else plt.cm.viridis((v + 45) / 90) for v in a]
    ax.add_collection(PolyCollection(corners(s), facecolors=cols, edgecolors='#333333', linewidths=0.3))
    if mark is not None and mark.any():
        ax.add_collection(PolyCollection(corners(s[mark]), facecolors='none', edgecolors='black', linewidths=1.4))
    ax.set_xlim(0, L); ax.set_ylim(0, L); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)


def load_packings(path):
    g = defaultdict(list)
    for r in csv.DictReader(open(path)):
        g[int(r['n'])].append((int(r['square_id']), float(r['x']), float(r['y']), float(r['angle'])))
    return {n: normalise([v[1:] for v in sorted(rows)]) for n, rows in g.items()}


def write_cfg(path, s):
    s = normalise(s)
    with open(path, 'w') as f:
        f.write(f'{len(s)} {side(s) + 1e-9:.17f}\n')
        for x, y, t in s:
            f.write(f'{x:.17f} {y:.17f} {math.degrees(t) % 90:.15f}\n')


def read_cfg(path):
    rows = [l.split() for l in open(path) if l.strip() and not l.startswith('#')]
    n = int(rows[0][0])
    return normalise([[float(a), float(b), math.radians(float(c))] for a, b, c, *_ in rows[1:1 + n]])


def quench(s, seconds=180, loosen=1.0, tag='q'):
    """One `fq quench --finish`. None if fq gives nothing back or the result overlaps."""
    a, b = str(WORK / f'{tag}_in.cfg'), str(WORK / f'{tag}_out.cfg')
    write_cfg(a, s)
    if os.path.exists(b):
        os.remove(b)
    subprocess.run([FQ, 'quench', '--in', a, '--out', b, '--finish', '--deadline', str(seconds), '--loosen', str(loosen)],
                   capture_output=True, timeout=seconds + 120)
    if not os.path.exists(b):
        return None
    q = read_cfg(b)
    return q if len(q) == len(s) and overlaps(q) == 0 else None


demo = load_packings(DATA / 'demo_packings.csv')      # the packings as they were when I stretched them
ours = pd.read_csv(DATA / 'ours_L.csv', index_col='n')  # my side per puzzle: previous notebook (L_v10) and this file

start = read_cfg(DATA / 'stretch_930_to_993_start.cfg')
after = read_cfg(DATA / 'stretch_930_to_993_fq.cfg')
fig, axs = plt.subplots(1, 3, figsize=(16, 5.6))
draw(axs[0], demo[930], f'n = 930, side {side(demo[930]):.6f}')
draw(axs[1], start, f'one column + one row copied in: {len(start)} squares, {overlaps(start)} overlapping pairs',
     mark=np.arange(len(start)) >= len(demo[930]))
draw(axs[2], after, f'993 after fq: {side(after):.6f}  (I had {ours.L_v10[993]:.6f})')
plt.tight_layout(); plt.show()

## Which packings are worth stretching

For K ≥ 20, the best side per puzzle in lostdecimal27's bank (29 Oct, most of them held by Cats and boxes) lines up on a single curve if you plot the overhang f = L − K against g = (D + 4)/K, where n = K² + D. The +4 came out of a fit; with it the points hardly scatter around the curve. My packings follow the same shape but sit higher. A few of them are on the curve or very close to it, and 744 and 930 are two of those. They share one design: two tilted arms of different thickness along two walls, ending against a block of upright squares in the corner, with the arm ends tapered.

In [ ]:
bank = pd.read_csv(BANK, dtype={'L': str})
lead = bank[bank.team.astype(str).str.startswith('Cats')]


def curve_points(ns, Ls):
    ns = np.asarray(ns, int); Ls = np.asarray(Ls, float)
    K = np.floor(np.sqrt(ns)).astype(int)
    D = ns - K * K
    f, g = Ls - K, (D + 4) / K
    keep = (K >= 20) & (D > 0) & (f < 0.98)
    return ns[keep], g[keep], f[keep]


ln, lg, lf = curve_points(lead.n.values, lead.L.astype(float).values)
edges = np.arange(0.10, 1.40, 0.05)
cen, med = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (lg >= lo) & (lg < hi)
    if m.sum() >= 3:
        cen.append(lg[m].mean()); med.append(np.median(lf[m]))
F = lambda g: np.interp(g, cen, med)       # the curve: bin medians, linear in between

on, og, of = curve_points(ours.index.values, ours.L_v10.values)
nn, ng, nf = curve_points(ours.index.values, ours.L_now.values)
print(f'Cats and boxes, K >= 20: {len(ln)} puzzles, scatter around the curve (std) {np.std(lf - F(lg)):.4f}')
print(f'mine above the curve, mean over K >= 20: previous file {np.mean(of - F(og)):+.4f}, this file {np.mean(nf - F(ng)):+.4f}')
for n in (744, 930):
    i = np.where(on == n)[0][0]
    print(f'n = {n}: g {og[i]:.3f}, f {of[i]:.4f}, curve {F(og[i]):.4f}')

won = set(pd.read_csv(DATA / 'stretch_wins.csv').n)
fig, ax = plt.subplots(figsize=(11, 6))
gg = np.linspace(min(cen), max(cen), 200)
ax.plot(gg, F(gg), color='0.5', lw=1, zorder=1)
ax.scatter(og, of, s=9, color='#f0ad4e', alpha=0.6, label='mine, previous file', zorder=2)
m = np.isin(nn, list(won))
ax.scatter(ng[m], nf[m], s=14, color='#1f77b4', label='mine, stretched (this file)', zorder=3)
ax.scatter(lg, lf, s=9, color='black', label='best per puzzle (Cats and boxes)', zorder=4)
for n in (744, 930):
    i = np.where(on == n)[0][0]
    ax.annotate(str(n), (og[i], of[i]), xytext=(og[i] - 0.06, of[i] + 0.06), arrowprops=dict(arrowstyle='-', lw=0.6))
ax.set_xlabel('g = (D + 4) / K'); ax.set_ylabel('f = L - K'); ax.legend(loc='upper left')
ax.set_title('K >= 20')
plt.tight_layout(); plt.show()

I couldn't build those tapered ends from scratch: my best assembler for this layout comes up three or four squares short of 930. So instead of rebuilding them, I copy them.

## The stretch

One vertical line cuts through the horizontal arm and the upright block, one horizontal line through the vertical arm and the block. To go from K to K + 1, I copy the column of arm squares next to each cut together with one column of the upright block, and push everything beyond the cut along by the arm's measured pitch instead of by 1 (1.03 and 1.05 for the two arms of 930). Deleting is the same in reverse. Where the copied pieces meet, a few squares overlap; `fq` (the quench from Evan Daniel's square-packing) takes it from there.

The code is in the hidden cell below. This is 930 → 993 run here:

In [ ]:
%%writefile /tmp/stretch.py
"""Stretch a packing by whole arm columns / rows (Squares in Squares).

A packing with tilted arms along the walls is cut with one vertical line (through a horizontal arm and the upright
block) and one horizontal line (through a vertical arm and the block). Insert (+1): copy the arm stack at the cut and
one upright column / row of the block next to it, and shift everything beyond the cut by the arm's measured pitch.
Delete (-1): remove them; arm squares beyond the cut move back by the pitch, everything else by 1. K -> K +- 1 per
(x, y) pair of operations; fq then relaxes the result.
Packings are arrays of (x, y, angle in radians).
"""
import math

import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree


def patches(a):
    """Connected groups (>= 15 squares) of tilted squares, with their lattice indices along / across the tilt."""
    deg = np.degrees(np.mod(a[:, 2], math.pi / 2)); deg = np.where(deg > 45, deg - 90, deg)
    t = np.where(np.abs(deg) > 3)[0]
    res = []
    if len(t) == 0:
        return res
    P = a[t, :2]
    pr = cKDTree(P).query_pairs(1.6, output_type='ndarray')
    A = coo_matrix((np.ones(len(pr)), (pr[:, 0], pr[:, 1])), shape=(len(P), len(P))) if len(pr) else coo_matrix((len(P), len(P)))
    nc, lab = connected_components(A, directed=False)
    for k in range(nc):
        idx = t[lab == k]
        if len(idx) < 15:
            continue
        th = math.radians(float(np.median(deg[idx])))
        u = np.array([math.cos(th), math.sin(th)]); v = np.array([-math.sin(th), math.cos(th)])
        pu, pv = a[idx, :2] @ u, a[idx, :2] @ v
        iu = np.round(pu - np.mean(pu - np.round(pu))).astype(int)
        iv = np.round(pv - np.mean(pv - np.round(pv))).astype(int)
        horiz = np.ptp(a[idx, 0]) >= np.ptp(a[idx, 1])
        res.append(dict(idx=idx, th=th, iu=iu, iv=iv, horiz=horiz,
                        x0=a[idx, 0].min(), x1=a[idx, 0].max(), y0=a[idx, 1].min(), y1=a[idx, 1].max()))
    return res


def stack_info(a, p, axis):
    """Stack index of every patch square across the cut, and the pitch: median centre distance between neighbouring
    stacks along the axis, measured on the middle half of the patch."""
    k = p['iu'] if axis == 0 else p['iv']
    coord = a[p['idx'], axis]
    ks = np.unique(k)
    means = np.array([coord[k == q].mean() for q in ks])
    mid = slice(len(ks) // 4, max(len(ks) // 4 + 2, 3 * len(ks) // 4))
    d = np.diff(means)[mid]
    pitch = float(np.median(np.abs(d))) if len(d) else 1.0
    return k, ks, means, pitch


def apply(a, axis, cut, op):
    """One insert (+1) or delete (-1) across `axis` (0: vertical cut at x = cut, 1: horizontal cut at y = cut)."""
    P = patches(a)
    cross = [p for p in P if (p['x0'] < cut < p['x1'] if axis == 0 else p['y0'] < cut < p['y1'])]
    good = [p for p in cross if p['horiz'] == (axis == 0)]
    if len(cross) != len(good) or not good:
        return None, 'cut crosses a patch along its length' if cross else 'no patch at the cut'
    in_patch = np.zeros(len(a), bool)
    beyond = a[:, axis] > cut
    copy_mask = np.zeros(len(a), bool)
    shift = np.zeros(len(a))
    pitches = []
    for p in good:
        k, ks, means, pitch = stack_info(a, p, axis)
        pitches.append(pitch)
        kc = ks[np.argmin(np.abs(means - cut))]
        in_patch[p['idx']] = True
        beyond[p['idx']] = k > kc
        copy_mask[p['idx'][k == kc]] = True
        if op < 0:
            shift[p['idx'][k > kc]] = -pitch
    d = max([1.0] + pitches)
    up_col = (~in_patch) & (a[:, axis] > cut - 1.0) & (a[:, axis] <= cut)
    deg = np.degrees(np.mod(a[:, 2], math.pi / 2)); upright = np.minimum(deg, 90 - deg) < 3
    up_col &= upright
    b = a.copy()
    if op > 0:
        b[beyond, axis] += d
        cp = a[copy_mask | up_col].copy()
        cp[:, axis] += d
        b = np.vstack([b, cp])
    else:
        rest = beyond & ~in_patch
        shift[rest] = -1.0
        b[:, axis] += shift
        b = b[~(copy_mask | up_col)]
    return b, f'axis {axis} cut {cut:.2f} op {op:+d} pitch {d:.4f} copied/removed {int((copy_mask | up_col).sum())}'


def pick_cuts(a, axis):
    """Cut through the middle of the largest horizontal (axis 0) / vertical (axis 1) arm, on an integer line of the
    upright block, without cutting another patch along its length."""
    P = [p for p in patches(a) if p['horiz'] == (axis == 0)]
    if not P:
        return None
    p = max(P, key=lambda q: len(q['idx']))
    lo, hi = (p['x0'], p['x1']) if axis == 0 else (p['y0'], p['y1'])
    c = math.floor((lo + hi) / 2) + 0.0
    for dc in (0, 1, -1, 2, -2, 3, -3):
        cc = c + dc
        others = [q for q in patches(a) if q['horiz'] != (axis == 0) and
                  ((q['x0'] - 0.8 < cc < q['x1'] + 0.8) if axis == 0 else (q['y0'] - 0.8 < cc < q['y1'] + 0.8))]
        if not others and lo + 2 < cc < hi - 2:
            return cc
    return None


def stretch(a, steps):
    """K -> K + steps (steps < 0 deletes). Returns (packing, notes) or (None, reason)."""
    b, notes = np.array(a, float), []
    for _ in range(abs(steps)):
        for axis in (0, 1):
            c = pick_cuts(b, axis)
            if c is None:
                return None, f'no cut on axis {axis}'
            b2, note = apply(b, axis, c, 1 if steps > 0 else -1)
            if b2 is None:
                return None, note
            b = b2; notes.append(note)
    return b, notes

In [ ]:
import stretch                                   # the file written above

b, notes = stretch.stretch(demo[930], +1)
print('\n'.join(notes))
print(f'{len(b)} squares, side {side(b):.6f} before fq, {overlaps(b)} overlapping pairs')
t0 = time.time()
q = quench(b, seconds=180, loosen=1.0, tag='s993')
print(f'after fq ({time.time() - t0:.0f} s): ' + (f'{side(q):.9f}' if q is not None else 'no valid result') +
      f'   my previous 993: {ours.L_v10[993]:.9f}', flush=True)

On my machine (one 180-second quench) this gave 31.924792 against the 31.932818 I had. `fq` stops at a wall-clock deadline, so the last digits depend on the machine.

Going the other way is a useful check. 930 minus one column and one row is 869, and 744 plus one is 800, both puzzles that I had built from the same design earlier. If the stretch is doing what it should, they come back at exactly the sides I already had:

In [ ]:
for n0, steps, n1 in ((930, -1, 869), (744, +1, 800)):
    b, notes = stretch.stretch(demo[n0], steps)
    q = quench(b, seconds=120, loosen=1.0, tag=f'c{n1}')
    print(f'{n0} {steps:+d} -> {len(b)} squares: fq {side(q) if q is not None else float("nan"):.9f}, '
          f'my {n1} {demo[n1].shape[0]} squares at {side(demo[n1]):.9f}', flush=True)

To try it on your own file: `stretch.stretch(packing, steps)` takes any packing as an array of (x, y, angle in radians) and returns the stretched packing, or the reason it couldn't cut it. It needs a tilted arm along a wall that is long enough to cut through the middle; packings without one are skipped.

## How far it goes

I ran it in batches against my file at the time and kept a stretched packing only where it beat the side I had.

In [ ]:
bt = pd.read_csv(DATA / 'stretch_batches.csv')
bt['what'] = bt['what'].str.replace('+-', '±', regex=False)
display(bt.drop(columns='batch').rename(columns={'what': 'batch', 'variants': 'stretched packings tried',
                                                 'wins': 'puzzles improved', 'score_gain': 'score gain'}))
w = pd.read_csv(DATA / 'stretch_wins.csv')
print(f'{len(w)} puzzles improved, {w.score_gain.sum():.6f} in score; biggest gains:')
top = w.sort_values('score_gain', ascending=False).head(12)
display(top[['n', 'K', 'D', 'specimen', 'steps', 'L_before', 'L_now', 'side_gain']].reset_index(drop=True))

The first batch took the packings closest to the curve and tried one and two steps each way; the second took every other packing with arms, one step each way. Then the winners became specimens themselves, which gave one more real win (932 → 995). Three and four steps found nothing. My reading: the copied arm keeps its thickness and angle, and three steps change the box enough that a different arm would fit better.

The four biggest moves, as they are in the file now:

In [ ]:
subdf = pd.read_csv(DATA / 'submission.csv')
show = w.sort_values('side_gain', ascending=False).n.head(4).tolist()
fig, axs = plt.subplots(1, 4, figsize=(18, 4.9))
for ax, n in zip(axs, show):
    s = normalise(subdf[subdf.n == n].sort_values('square_id')[['x', 'y', 'angle']].astype(float).values)
    draw(ax, s, f'n = {n}: {side(s):.6f}  (was {ours.L_v10[n]:.6f})')
plt.tight_layout(); plt.show()

## Stretches that almost won

In those batches every stretched packing got two quenches of up to three minutes. For the ones that lost narrowly that was too little: a fresh stretch is far less relaxed than a packing that has been polished for days. So the stretches that lost by less than 0.01 got a longer chain: quench with a small random kick (σ between 0.002 and 0.02) and a random loosen, keep the result whenever the side drops, thirty steps. Seven turned into wins. Forty more steps for the chains that had won or come within 0.003 moved seven puzzles again, by about the same amount, so the chains had not stopped improving. A longer run is going now.

In [ ]:
rs = pd.read_csv(DATA / 'relax_steps.csv')
display(rs.rename(columns={'puzzles': 'puzzles improved', 'score_gain': 'score gain'}))
print(f'together: {rs.score_gain.sum():.6f}')

The same chain found a win even among stretches that had lost by 0.03 to 0.05 (n = 995), so I no longer throw those away.

## What didn't work

I expected the #1 team's published packings to stretch at least as well as mine. They lost everywhere, and the reason was simple: their published 299 is 17.8133, worse than both their leaderboard entry (17.7936) and my 17.8012. Those are older versions.

I also tried two things that only need more computer time. Annealing a whole box with replica exchange (n = 262, 32 temperatures, six minutes) went nowhere: neighbouring replicas swapped in at most 1 % of the attempts, and the coldest one stayed stuck with overlaps. A packing this dense doesn't melt and re-form, and my estimate is that it would take hundreds of temperatures. Then a local version: freeze the packing, add one square inside a window, and anneal only the window (8 puzzles, 32 windows). Putting back a square I had deleted worked every time; adding a new one never did. The empty space in a window of radius 2.5 adds up to only 2.4 to 3.7 squares, and no gap in it is wider than about 0.6.

Ten hours of annealing with `fq` inside (on Kaggle) took the published 299 from 17.8133 to 17.8027, still above my 17.8012, and moved my own 299 and 232 by about 2e-4 each. Useful polish, no new structure.

## Before this

The rest of the file is the public frontier plus what my earlier notebook added: an L-band builder, re-quench rounds, and alex chilton's fault-line DP run with my file as a donor. That notebook explains those parts: [L-band builds, re-quench, and the #1 packings](https://www.kaggle.com/code/az05192000gmailcom/l-band-builds-re-quench-and-the-1-packings).

## Submission

Besides the stretches, the file takes 52 puzzles from public notebooks that were ahead of mine this morning, all but one by less than 1e-5 in side; n = 852 from prvsiyan's notebook is 0.0021 better. The packings are in the attached dataset, and the cell below scores the file with the official metric after a pandas float64 read, the way the leaderboard reads it.

In [ ]:
shutil.copy(DATA / 'submission.csv', WORK / 'submission.csv')
import sis_metric as M
M.KAGGLE_CORES = len(os.sched_getaffinity(0))
t0 = time.time()
kw = {'float_precision': 'round_trip'} if int(pd.__version__.split('.')[0]) < 3 else {}   # pandas 3 reads floats exactly by default
sdf = pd.read_csv(WORK / 'submission.csv', **kw)
total, results = M._evaluate_puzzles_parallel(M._parse_submission_csv(sdf.drop(columns=['id']).to_csv(index=False)),
                                              set(range(1, M.MAX_N + 1)))
bad = [n for n, _, _, st in results if st != 'ok']
print(f'official metric: {total:.10f}, rejected puzzles: {len(bad)}   ({time.time() - t0:.0f} s)')

for p in WORK.glob('*.cfg'):
    p.unlink()

## Credits

- `fq`: Evan Daniel, [square-packing](https://github.com/evand/square-packing) (MIT), included in the dataset as a Linux binary built from the unmodified source (build command in `LICENSE_fq.txt`).
- The per-puzzle bests in the plot: lostdecimal27's [per-puzzle bank](https://www.kaggle.com/datasets/lostdecimal27/squares-in-squares-bank).
- The #1 team's published packings: pjmathematician, [Three square packings that beat the records](https://www.kaggle.com/code/pjmathematician/three-square-packings-that-beat-the-records).
- Public packings merged on 6 Nov: prvsiyan's [A Square Deal](https://www.kaggle.com/code/prvsiyan/a-square-deal-research-to-verified-packings) (852), Malyshev Danil's Living Atlas, Kirill Tymchishin, jazivxt and sigmaborov (small moves).
- The public packings in the rest of the file, as credited in [my previous notebook](https://www.kaggle.com/code/az05192000gmailcom/l-band-builds-re-quench-and-the-1-packings): mostly Malyshev Danil's [Living Atlas](https://www.kaggle.com/code/malyshevdanil/squares-in-squares-living-atlas-of-tight-packings), Kirill Tymchishin, sigmaborov, alex chilton, jazivxt, SeshuRaju, and everyone they build on.